# Bagging and Random Forests
A deep tree has low bias but high variance. I want to test whether averaging many decorrelated trees improves held-out performance.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

X,y=load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
models={
 'tree':DecisionTreeClassifier(random_state=42),
 'bagging':BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=150,random_state=42),
 'forest':RandomForestClassifier(n_estimators=250,max_features='sqrt',oob_score=True,n_jobs=-1,random_state=42),
}
rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]
    rows.append({'model':name,'train_acc':m.score(Xtr,ytr),'test_auc':roc_auc_score(yte,p),'oob':getattr(m,'oob_score_',None)})
pd.DataFrame(rows).round(4)


## Why it works
Bagging averages bootstrap-trained trees, reducing variance. Random forests also sample candidate features at each split, making trees less correlated. Averaging correlated errors helps less than averaging diverse errors.